# Paper-Based IMC Workflow

## Step 3: Prepare the Steinbock + MESMER segmentation handoff

This notebook moves from composite design into a **tool-faithful Steinbock project handoff**.

We still stop before downstream quantification and phenotyping. In this step, our goal is narrower and very important:

- create a Steinbock-style project layout
- create a panel file that tells Steinbock how to group channels for DeepCell / MESMER
- document exactly how the adapted channel mapping differs from the paper
- prepare the command lines needed to run segmentation

This is the bridge between our adapted biological reasoning and the actual segmentation tooling.

## What the official tools expect

According to the Steinbock documentation, DeepCell / MESMER segmentation in Steinbock works as follows:

- `steinbock segment deepcell --minmax` runs whole-cell segmentation using DeepCell / MESMER
- Steinbock expects a **two-channel DeepCell input** for Mesmer whole-cell segmentation
- the **first channel** must be a nuclear channel
- the **second channel** must be a membrane or cytoplasmic channel
- if the panel file contains a `deepcell` column, Steinbock groups channels by that column and aggregates each group into one channel
- the default aggregation can be changed, but mean aggregation is the default Steinbock behavior for grouped channels

This is exactly what we want, because it means we do **not** need to manually build a 2-channel file if we provide a correct Steinbock panel table. Instead, Steinbock can generate the DeepCell input from the raw channel images and our grouping instructions.

That is much closer to the paper's workflow than bypassing Steinbock entirely.

## How this maps to our adapted panel

The paper's segmentation inputs were:

- nuclear: `HistoneH3 + 191Ir + 193Ir`
- membrane: `CD98 + CD3 + CD138 + CD45`

For our ROI, the adapted mapping is:

### DeepCell nuclear group
- `DNA1` (`191Ir` intercalator)
- `DNA2` (`193Ir` intercalator)

### DeepCell membrane group
- `CD3`
- `CD138`
- `CD31`
- `aSMA`

This means our panel file will explicitly assign:

- `deepcell = nuclear` for `DNA1` and `DNA2`
- `deepcell = membrane` for `CD3`, `CD138`, `CD31`, and `aSMA`
- empty `deepcell` values for all other markers

That way, Steinbock will ignore unrelated channels when constructing the MESMER input.

## What this step will do

This notebook performs only the following tasks:

1. create a clean Steinbock project folder
2. copy the ROI TIFF files into an `images/` directory
3. generate a panel CSV with the `deepcell` grouping column
4. write a metadata note documenting the adapted segmentation design
5. check whether Docker is available locally for later Steinbock execution
6. generate the exact command lines for segmentation, but stop before any downstream measurement or normalization

This notebook is intentionally still a setup step. Once the project structure is ready and reviewed, the next decision will be whether to actually run the segmentation command on your system.

In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess

WORKFLOW_ROOT = Path('/Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow')
ROI_DIR = Path('/Users/rashid/1_IMC_Analysis/11_Vincenzo/ROI001_D13')

STEP3_ROOT = WORKFLOW_ROOT / 'step3_steinbock_project'
PROJECT_ROOT = STEP3_ROOT / ROI_DIR.name
IMAGES_DIR = PROJECT_ROOT / 'images'
MASKS_DIR = PROJECT_ROOT / 'masks'
PANEL_PATH = PROJECT_ROOT / 'panel.csv'
METADATA_PATH = PROJECT_ROOT / 'adaptation_metadata.json'
COMMANDS_PATH = PROJECT_ROOT / 'run_commands.txt'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)

NUCLEAR_GROUP = ['DNA1', 'DNA2']
MEMBRANE_GROUP = ['CD3', 'CD138', 'CD31', 'aSMA']

print('Project root:', PROJECT_ROOT)
print('Images dir:', IMAGES_DIR)
print('Panel path:', PANEL_PATH)


Project root: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13
Images dir: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/images
Panel path: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/panel.csv


## Building the Steinbock image layout

Steinbock works from a project directory containing the raw per-channel images. For this step, we place the ROI TIFF files into an `images/` folder under a dedicated project root.

This does two things:

- it isolates the paper-based workflow from the rest of the workspace
- it creates a standard place for Steinbock to look for images and later write masks

We are copying the files here rather than moving them, so the original ROI folder remains untouched.

In [2]:
channel_files = sorted(list(ROI_DIR.glob('*.tif')) + list(ROI_DIR.glob('*.tiff')))
for src in channel_files:
    dst = IMAGES_DIR / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

print(f'Copied {len(channel_files)} files into {IMAGES_DIR}')


Copied 43 files into /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/images


## Generating the panel table

The panel table is the key technical object in this step.

We will create a CSV that records, for each channel image:

- the metal tag
- the marker name
- the original filename
- the `deepcell` group assignment

The important part is the `deepcell` column:

- `nuclear` for `DNA1` and `DNA2`
- `membrane` for `CD3`, `CD138`, `CD31`, `aSMA`
- blank for all other channels

This gives Steinbock the exact instructions it needs to construct the 2-channel Mesmer input automatically.

In [3]:
def parse_channel(path: Path) -> dict:
    stem = path.name.replace('.ome.tiff', '').replace('.ome.tif', '')
    if '_' in stem:
        metal_tag, marker = stem.split('_', 1)
    else:
        metal_tag, marker = 'UNKNOWN', stem
    return {'metal_tag': metal_tag, 'marker': marker, 'filename': path.name}


rows = []
for path in sorted(IMAGES_DIR.glob('*.tif')) + sorted(IMAGES_DIR.glob('*.tiff')):
    row = parse_channel(path)
    if row['marker'] in NUCLEAR_GROUP:
        row['deepcell'] = 'nuclear'
    elif row['marker'] in MEMBRANE_GROUP:
        row['deepcell'] = 'membrane'
    else:
        row['deepcell'] = ''
    rows.append(row)

with PANEL_PATH.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['metal_tag', 'marker', 'filename', 'deepcell'])
    writer.writeheader()
    writer.writerows(rows)

print(f'Panel written to: {PANEL_PATH}')
print('Nuclear assignments:', [r['marker'] for r in rows if r['deepcell'] == 'nuclear'])
print('Membrane assignments:', [r['marker'] for r in rows if r['deepcell'] == 'membrane'])


Panel written to: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/panel.csv
Nuclear assignments: ['DNA1', 'DNA2']
Membrane assignments: ['CD138', 'CD31', 'CD3', 'aSMA']


## Writing an adaptation record

A paper-based workflow should not rely on memory to explain why a specific panel mapping was used.

So in addition to the panel CSV, we save a JSON metadata record that explains:

- the paper's original segmentation channels
- the available channels in this ROI
- the adapted Steinbock / DeepCell grouping used here
- the scientific reason for the adaptation

This is useful for later methods writing, reproducibility, and collaboration.

In [4]:
metadata = {
    'roi_id': ROI_DIR.name,
    'workflow': 'paper_based_adapted_steinbock_mesmer',
    'reference_paper_channels': {
        'nuclear': ['HistoneH3', '191Ir', '193Ir'],
        'membrane': ['CD98', 'CD3', 'CD138', 'CD45']
    },
    'roi_available_equivalents': {
        '191Ir': 'DNA1',
        '193Ir': 'DNA2',
        'HistoneH3': None,
        'CD98': None,
        'CD45': None
    },
    'deepcell_grouping': {
        'nuclear': NUCLEAR_GROUP,
        'membrane': MEMBRANE_GROUP
    },
    'rationale': [
        'DNA1 and DNA2 correspond to the available 191Ir and 193Ir intercalator channels.',
        'HistoneH3 is not present in this ROI export.',
        'CD3 and CD138 overlap with the paper membrane design.',
        'CD31 and aSMA were added to strengthen structural boundary information in sarcoma tissue.',
        'This is an adapted, documented Steinbock + MESMER input design and not an exact channel-level reproduction of the paper.'
    ]
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Metadata written to: {METADATA_PATH}')


Metadata written to: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/adaptation_metadata.json


## Checking execution prerequisites

Before running Steinbock, we should check whether the local environment has the basic prerequisites available.

For the Steinbock Docker workflow, the practical minimum is:

- Docker installed and callable from the terminal
- permission to run containers locally

If Docker is not available, that does **not** mean the workflow is invalid. It simply means the segmentation execution must be moved to a machine or environment where Steinbock can run.

In [5]:
docker_path = shutil.which('docker')
print('Docker executable:', docker_path)

docker_ok = False
docker_message = None
if docker_path is not None:
    try:
        result = subprocess.run(
            ['docker', '--version'],
            capture_output=True,
            text=True,
            check=False,
        )
        docker_ok = result.returncode == 0
        docker_message = (result.stdout or result.stderr).strip()
    except Exception as exc:
        docker_message = str(exc)

print('Docker available:', docker_ok)
print('Docker message:', docker_message)


Docker executable: /usr/local/bin/docker
Docker available: True
Docker message: Docker version 29.2.1, build a5c7197


## Preparing the run commands

At this stage, we write the commands needed for segmentation, but we do not yet assume that they will run successfully on this machine.

The key segmentation command in Steinbock is:

`steinbock segment deepcell --minmax`

The `--minmax` option is consistent with Steinbock's documented channel-wise normalization for DeepCell segmentation input.

In practice, there are two ways this may be used later:

- inside a Steinbock Docker container
- on a local setup where Steinbock is already installed

For now, we generate both a human-readable record and a command block you can reuse later.

In [6]:
project_path_str = str(PROJECT_ROOT)

commands = [
    '# Run from the Steinbock project root',
    f'cd {project_path_str}',
    '',
    '# If steinbock is installed locally',
    'steinbock segment deepcell --minmax',
    '',
    '# The project expects the panel.csv in the project root and the TIFF channel images in images/',
    '# Steinbock will construct the 2-channel DeepCell input using the deepcell column in panel.csv.',
]

COMMANDS_PATH.write_text('\n'.join(commands), encoding='utf-8')
print(f'Commands written to: {COMMANDS_PATH}')
print('\n'.join(commands))


Commands written to: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13/run_commands.txt
# Run from the Steinbock project root
cd /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step3_steinbock_project/ROI001_D13

# If steinbock is installed locally
steinbock segment deepcell --minmax

# The project expects the panel.csv in the project root and the TIFF channel images in images/
# Steinbock will construct the 2-channel DeepCell input using the deepcell column in panel.csv.


## How to review Step 3

Before moving on, check the following carefully:

### 1. Does the panel file reflect the adapted biology correctly?
Open `panel.csv` and confirm that only `DNA1`, `DNA2`, `CD3`, `CD138`, `CD31`, and `aSMA` are assigned to the `deepcell` groups.

### 2. Does the metadata file explain the adaptation honestly?
The metadata should make it obvious that this is paper-based but adapted, not a hidden deviation.

### 3. Is Docker available on the machine where you want to run segmentation?
If not, the project is still ready, but execution will need to happen elsewhere.

### 4. Why we stop here
Once segmentation is actually run, the workflow will start creating masks that become the basis for all later measurement. That is a major transition point, so it is worth reviewing the setup before execution.

## Stop point

This notebook stops after preparing the Steinbock + MESMER project handoff.

The next step would be to actually run segmentation and inspect the produced masks, but we should only do that after confirming that this project layout and panel definition look correct to you.